# 🤖 Autonomous Agentic Study Assistant

Welcome to your **Agentic AI** study companion! Unlike standard chatbots that only generate text from memory, this **Agent** can autonomously plan and use external tools:

1. 🔍 **`search_wikipedia`**: Queries live Wikipedia articles to fetch accurate, factual knowledge.
2. 🧮 **`calculate`**: Solves math, physics, and scientific formulas without hallucinations.
3. 📝 **`save_study_notes`**: Automatically compiles and writes revision guides, summaries, or flashcards directly to disk in `study_notes/`.

In [1]:
import os
import sys
import json
import math
from pathlib import Path
import httpx
from dotenv import load_dotenv
from groq import Groq

# Ensure UTF-8 output on Windows
if sys.platform == "win32":
    sys.stdout.reconfigure(encoding="utf-8")

# 1. Load API Key from .env
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
if not api_key or "your_" in api_key:
    raise ValueError("Please set your GROQ_API_KEY inside the .env file!")

client = Groq(api_key=api_key)
MODEL_NAME = "openai/gpt-oss-120b"
print(f"✅ Groq Client initialized with Agent model: {MODEL_NAME}")

✅ Groq Client initialized with Agent model: openai/gpt-oss-120b


## 🛠️ Step 1: Define Tools (The Agent's Capabilities)

In [2]:
def search_wikipedia(query: str) -> str:
    """Searches Wikipedia and returns top summaries for study research."""
    try:
        headers = {"User-Agent": "AgenticStudyBot/1.0 (contact@studybot.ai)"}
        res = httpx.get(
            "https://en.wikipedia.org/w/api.php",
            params={"action": "query", "list": "search", "srsearch": query, "format": "json"},
            headers=headers,
            timeout=10.0
        )
        data = res.json()
        search_items = data.get("query", {}).get("search", [])
        if not search_items:
            return f"No Wikipedia articles found for '{query}'."
        
        summaries = []
        for item in search_items[:2]:
            title = item["title"]
            r = httpx.get(
                f"https://en.wikipedia.org/api/rest_v1/page/summary/{title}",
                headers=headers,
                timeout=10.0
            )
            if r.status_code == 200:
                extract = r.json().get("extract", "")
                summaries.append(f"Title: {title}\nSummary: {extract}")
        
        return "\n\n".join(summaries) if summaries else "No article content available."
    except Exception as e:
        return f"Error searching Wikipedia: {str(e)}"

def calculate(expression: str) -> str:
    """Safely evaluate mathematical and scientific expressions."""
    safe_math = {
        "sqrt": math.sqrt, "sin": math.sin, "cos": math.cos, "tan": math.tan,
        "log": math.log, "log10": math.log10, "exp": math.exp, "pi": math.pi, "e": math.e
    }
    try:
        cleaned_expr = expression.replace("^", "**")
        result = eval(cleaned_expr, {"__builtins__": {}}, safe_math)
        return str(result)
    except Exception as e:
        return f"Calculation error: {str(e)}"

def save_study_notes(filename: str, content: str) -> str:
    """Saves study notes, cheat sheets, or flashcards into the 'study_notes' directory."""
    try:
        notes_dir = Path("study_notes")
        notes_dir.mkdir(exist_ok=True)
        safe_filename = Path(filename).name
        if not safe_filename.endswith((".txt", ".md", ".json")):
            safe_filename += ".md"
        filepath = notes_dir / safe_filename
        filepath.write_text(content, encoding="utf-8")
        return f"Successfully saved study note to '{filepath.as_posix()}'."
    except Exception as e:
        return f"Error saving file: {str(e)}"

# Map function names to callables
AVAILABLE_TOOLS = {
    "search_wikipedia": search_wikipedia,
    "calculate": calculate,
    "save_study_notes": save_study_notes,
}

# Tool schemas for Groq Function Calling
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "search_wikipedia",
            "description": "Search Wikipedia for factual concepts, history, science, or literature.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search topic or keywords"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform mathematical or scientific calculations (supports sqrt, log, exponents, etc.).",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression (e.g. 'sqrt(144) * 5' or '1643 + 300')"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "save_study_notes",
            "description": "Save study notes, flashcards, or summary guides as a markdown/text file on disk.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {"type": "string", "description": "Name of the file, e.g. 'quantum_physics.md'"},
                    "content": {"type": "string", "description": "The full markdown text content to save"}
                },
                "required": ["filename", "content"]
            }
        }
    }
]
print("✅ Registered Tools: search_wikipedia, calculate, save_study_notes")

✅ Registered Tools: search_wikipedia, calculate, save_study_notes


## 🧠 Step 2: Agent ReAct Reasoning Loop

In [3]:
SYSTEM_PROMPT = (
    "You are an autonomous Agentic Study Assistant.\n"
    "Guidelines:\n"
    "1. When asked about factual, historical, or scientific topics, invoke 'search_wikipedia'.\n"
    "2. When calculations are involved, invoke 'calculate' to compute exact values.\n"
    "3. If the user asks to save or draft notes, invoke 'save_study_notes'.\n"
    "4. Provide concise, high-yield study guidance."
)

def run_study_agent(user_query: str, max_iterations: int = 5):
    """Autonomous agent loop handling reasoning, tool execution, and observation."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query}
    ]
    
    print(f"\n🎯 Goal: {user_query}\n" + "=" * 60)
    
    for step in range(1, max_iterations + 1):
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            tools=TOOL_SCHEMAS,
            tool_choice="auto",
            temperature=0.3
        )
        message = response.choices[0].message
        
        # Check if the agent decided to call one or more tools
        if message.tool_calls:
            messages.append(message)
            for tool_call in message.tool_calls:
                fn_name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)
                print(f"[Step {step}] 🛠️ Agent Action: {fn_name}({args})")
                
                tool_fn = AVAILABLE_TOOLS.get(fn_name)
                result = tool_fn(**args) if tool_fn else f"Error: Tool '{fn_name}' not found."
                
                preview = (result[:120] + "...") if len(result) > 120 else result
                print(f"[Step {step}] 👀 Observation: {preview}\n")
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })
        else:
            print("=" * 60)
            print(f"🎓 Final Answer:\n{message.content}\n")
            return message.content
            
    print("Reached maximum agent iterations.")

## 🚀 Step 3: Run the Agent on Autonomous Multi-Step Tasks

In [4]:
# Multi-step autonomous agent execution
run_study_agent(
    "Research Isaac Newton on Wikipedia, calculate what year it was 300 years after his birth (1643), "
    "and save a 3-bullet revision note to newton_facts.md"
)


🎯 Goal: Research Isaac Newton on Wikipedia, calculate what year it was 300 years after his birth (1643), and save a 3-bullet revision note to newton_facts.md
[Step 1] 🛠️ Agent Action: search_wikipedia({'query': 'Isaac Newton'})
[Step 1] 👀 Observation: Title: Isaac Newton
Summary: Sir Isaac Newton was an English polymath who was a mathematician, physicist, astronomer, alch...

[Step 2] 🛠️ Agent Action: calculate({'expression': '1643 + 300'})
[Step 2] 👀 Observation: 1943

[Step 3] 🛠️ Agent Action: save_study_notes({'content': '- Born 4 January 1643 (24 December 1642 OS) in Woolsthorpe, England.\n- Authored *Philosophiæ Naturalis Principia Mathematica* (1687), establishing the laws of motion and universal gravitation.\n- Developed calculus (independently of Leibniz) and made seminal contributions to optics; 300 years after his birth was **1943**.', 'filename': 'newton_facts.md'})
[Step 3] 👀 Observation: Successfully saved study note to 'study_notes/newton_facts.md'.

🎓 Final Answer:
The 

## 💬 Step 4: Interactive Agentic Study Session
Run this cell to prompt the agent with any custom study task (e.g., *"Explain escape velocity and calculate it for Earth"* or *"Summarize Photosynthesis and save notes"*).

In [ ]:
print("🎓 Agentic Study Assistant Ready! (Type 'quit' or 'exit' to end)\n")

while True:
    user_input = input("Enter your study goal: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ["quit", "exit", "q", "bye"]:
        print("\n👋 Great study session! Keep learning!")
        break
    run_study_agent(user_input)